# Lab 7: Word Sequences

In [1]:
from pathlib import Path
import tarfile

paths = [Path('../sts/resources/test-gold.tgz'), Path('sts/resources/test-gold.tgz'), Path('test-gold.tgz')]
path = next((p for p in paths if p.is_file()), None)
if path is None:
    raise FileNotFoundError('Could not find test-gold.tgz')

with tarfile.open(path) as tar:
    pairs = [line.split('\t') for line in tar.extractfile('test-gold/STS.input.SMTeuroparl.txt').read().decode('utf-8').splitlines()]
    gs = [float(line) for line in tar.extractfile('test-gold/STS.gs.SMTeuroparl.txt').read().decode('utf-8').splitlines()]

assert len(pairs) == len(gs)
assert all(len(pair) == 2 for pair in pairs)

unique_sentences = list(dict.fromkeys(sentence for pair in pairs for sentence in pair))
print(f'{len(pairs)} pairs; {len(unique_sentences)} unique sentences')

459 pairs; 338 unique sentences


## Words and named entities

We use NLTK's named-entity chunker. We keep capital letters and punctuation for tagging, then lowercase the words and drop punctuation for Jaccard. The word baseline uses the same regex as in Labs 3 and 6. For words plus named entities, each detected entity counts as one item, such as `john smith`.

The main version uses `binary=True`, so the chunker only decides whether a span is an entity. With entity types the same chunker cuts names into smaller pieces (for example `van:PERSON` and `orden report:PERSON`), so we keep the typed version only as a variant. We also try lemmas instead of words, lemmatizing with WordNet as in session 6 and leaving the entities untouched. We tokenize with a regex instead of `nltk.word_tokenize` to keep the word baseline of the previous sessions, although the tagger may handle a few tokens (such as contractions) worse.

In [2]:
import re

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tree import Tree

resources = {
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'maxent_ne_chunker_tab': 'chunkers/maxent_ne_chunker_tab/',
    'words': 'corpora/words',
    'wordnet': 'corpora/wordnet',
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

word_pattern = re.compile(r'\w+')
token_pattern = re.compile(r'\w+|[^\w\s]')
word_sets = {
    sentence: {word.casefold() for word in word_pattern.findall(sentence)}
    for sentence in unique_sentences
}

lemmatizer = WordNetLemmatizer()
wordnet_pos = {'N': 'n', 'V': 'v', 'J': 'a', 'R': 'r'}

def words_and_entities(tree, lemmas=False, types=False):
    items = []
    for node in tree:
        if isinstance(node, Tree):
            words = [word.casefold() for word, _ in node.leaves() if word_pattern.fullmatch(word)]
            if words:
                items.append(' '.join(words) + (f':{node.label()}' if types else ''))
        else:
            word, tag = node
            if word_pattern.fullmatch(word):
                word, pos = word.casefold(), wordnet_pos.get(tag[0])
                items.append(lemmatizer.lemmatize(word, pos) if lemmas and pos else word)
    return items

In [3]:
example = 'John Smith is working.'
example_tree = nltk.ne_chunk(nltk.pos_tag(token_pattern.findall(example)), binary=True)
print(words_and_entities(example_tree))

tagged_sentences = nltk.pos_tag_sents([token_pattern.findall(sentence) for sentence in unique_sentences])
chunked_sentences = list(nltk.ne_chunk_sents(tagged_sentences, binary=True))
typed_sentences = nltk.ne_chunk_sents(tagged_sentences, binary=False)
entity_sets = {
    sentence: set(words_and_entities(tree))
    for sentence, tree in zip(unique_sentences, chunked_sentences)
}
typed_entity_sets = {
    sentence: set(words_and_entities(tree, types=True))
    for sentence, tree in zip(unique_sentences, typed_sentences)
}
lemma_sets = {
    sentence: set(words_and_entities(tagged, lemmas=True))
    for sentence, tagged in zip(unique_sentences, tagged_sentences)
}
lemma_entity_sets = {
    sentence: set(words_and_entities(tree, lemmas=True))
    for sentence, tree in zip(unique_sentences, chunked_sentences)
}

['john smith', 'is', 'working']


## Comparison

The gold scores measure similarity, so we use the Jaccard coefficient (one minus Jaccard distance). We compare both sets of scores with the gold scores using Pearson correlation.

In [4]:
from statistics import correlation

from nltk.metrics import jaccard_distance

def similarities(sets):
    return [
        1 - jaccard_distance(sets[a], sets[b]) if sets[a] or sets[b] else 1.0
        for a, b in pairs
    ]

word_scores = similarities(word_sets)
entity_scores = similarities(entity_sets)

print('Pair | Gold | Words | Words + NEs')
for i in range(10):
    print(f'{i + 1} | {gs[i]:.2f} | {word_scores[i]:.3f} | {entity_scores[i]:.3f}')

representations = {
    'Words': word_sets,
    'Words + NEs': entity_sets,
    'Words + typed NEs': typed_entity_sets,
    'Lemmas': lemma_sets,
    'Lemmas + NEs': lemma_entity_sets,
}
print(f"\n{'Representation':<20}Pearson")
for name, sets in representations.items():
    print(f'{name:<20}{correlation(gs, similarities(sets)):.6f}')
changed = sum(word_score != entity_score for word_score, entity_score in zip(word_scores, entity_scores))
print(f'Similarity changed for {changed} of {len(pairs)} pairs')

Pair | Gold | Words | Words + NEs
1 | 4.50 | 0.333 | 0.333
2 | 5.00 | 0.769 | 0.750
3 | 4.25 | 0.381 | 0.381
4 | 4.50 | 0.583 | 0.583
5 | 5.00 | 1.000 | 1.000
6 | 5.00 | 0.368 | 0.368
7 | 4.67 | 0.294 | 0.294
8 | 5.00 | 0.750 | 0.750
9 | 5.00 | 0.286 | 0.286
10 | 5.00 | 0.500 | 0.500

Representation      Pearson
Words               0.485346
Words + NEs         0.488886
Words + typed NEs   0.488927
Lemmas              0.486806
Lemmas + NEs        0.488137
Similarity changed for 54 of 459 pairs


In [5]:
for pair_number in (57, 158):
    i = pair_number - 1
    print(f'Pair {pair_number} (gold: {gs[i]:.2f})')
    for sentence in pairs[i]:
        print(sentence)
        print('Multiword entities:', sorted(item for item in entity_sets[sentence] if ' ' in item))
    print(f'Words: {word_scores[i]:.3f}; words + NEs: {entity_scores[i]:.3f}\n')

Pair 57 (gold: 5.00)
Van Orden Report (A5-0241/2000)
Multiword entities: ['van orden report']
Van Orden report (A5-0241 / 2000)
Multiword entities: ['van orden']
Words: 1.000; words + NEs: 0.500

Pair 158 (gold: 3.00)
Van Orden Report (A5-0241/2000)
Multiword entities: ['van orden report']
Report Horsebox Orden (A5-0241 / 2000)
Multiword entities: ['report horsebox orden']
Words: 0.714; words + NEs: 0.600



## Justification

Words plus NEs give a correlation of 0.489, compared with 0.485 for words alone. Only 54 of the 459 pairs change their similarity, so the difference is very small. Compared with session 6, it is slightly above words and lemmas (0.485 and 0.483) and still clearly better than the sense-based versions (between 0.406 and 0.422). The values of session 6 were negative because there we used Jaccard distance instead of the coefficient.

The variants do not change this picture. Adding the entity type gives 0.489, the same as without types. Lemmas give 0.487 and lemmas plus NEs 0.488 (the lemma value is a bit higher than in session 6 because here the tagger also sees the punctuation). The five versions are within 0.004 of each other.

For this dataset, we would stay with words, because the NEs add an extra step and the result is practically the same. The effect of the NEs also goes in both directions. In pair 158, the sentences mention different names that share the words “report” and “orden”. As single items they no longer match, so the similarity goes from 0.714 to 0.600, closer to the gold score of 3. In pair 57, both sentences refer to the same report (gold 5), but the chunker takes “Van Orden Report” in one and only “Van Orden” in the other, because “report” is in lowercase. The similarity drops from 1.000 to 0.500, although the sentences are almost identical.

NEs can help when the sentences contain several multiword names and those names are what makes them different, as in pair 158, where the two names share words but are not the same entity. This requires that the recognizer gives the same boundaries to the same entity in both sentences. Here that does not always happen, because the NLTK chunker depends a lot on capital letters. When the sentences have few entities, or the same entity is written in different ways, words are more reliable.